<a href="https://colab.research.google.com/github/abhijadhav14/Data-Analytics-Using-Python/blob/main/GMM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Gaussian Mixture Model (GMM) using PySpark MLlib

# Import Libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import randn
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import GaussianMixture

# Initialize Spark Session
spark = SparkSession.builder.appName("GMMLargeData").getOrCreate()

# 1. Dataset (100,000 rows, 3 features)
df = spark.range(0, 100000).select(
    (randn(seed=10) * 20).alias("income"),
    (randn(seed=11) * 5).alias("age_scaled"),
    (randn(seed=12) * 1000).alias("spend_score")
)

# 2. Combine Features into a Single Vector
assembler = VectorAssembler(
    inputCols=["income", "age_scaled", "spend_score"],
    outputCol="raw_features"
)
df_assembled = assembler.transform(df)

# 3. Scale the Features
scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withStd=True,
    withMean=True
)
scaler_model = scaler.fit(df_assembled)
final_df = scaler_model.transform(df_assembled)

# 4. Create and Train GMM Model
# k=3 clusters. GMM is probabilistic, meaning it calculates the probability
# that a point belongs to a specific cluster.
gmm = GaussianMixture(featuresCol="features", predictionCol="cluster", probabilityCol="probability", k=3, seed=42)
gmm_model = gmm.fit(final_df)

# 5. Make Predictions
predictions = gmm_model.transform(final_df)

# 6. Show GMM Parameters (Weights, Means, and Covariances of the Gaussians)
print("Gaussian Mixture Model Parameters:")
for i in range(gmm_model.getK()):
    print(f"\nCluster {i}:")
    print(f"Weight: {gmm_model.weights[i]}")
    print(f"Mean: {gmm_model.gaussiansDF.select('mean').collect()[i][0]}")

# 7. View Example Predictions (Shows assigned cluster and the probability array)
print("\nSample Cluster Assignments with Probabilities:")
predictions.select("income", "spend_score", "cluster", "probability").show(5, truncate=False)

Gaussian Mixture Model Parameters:

Cluster 0:
Weight: 0.37272663191182653
Mean: [-0.0791982698981286,-0.14832682007666087,-0.2066597378404129]

Cluster 1:
Weight: 0.19084765590372693
Mean: [-0.4278532786266431,0.04624363659658028,0.30390205848101043]

Cluster 2:
Weight: 0.4364257121844466
Mean: [0.2547377402044684,0.10645538318972543,0.04360098871438136]

Sample Cluster Assignments with Probabilities:
+------------------+-------------------+-------+------------------------------------------------------------+
|income            |spend_score        |cluster|probability                                                 |
+------------------+-------------------+-------+------------------------------------------------------------+
|-9.646324578924691|-189.72856311959984|0      |[0.46205511713812186,0.22576881971024973,0.3121760631516284]|
|8.944628280207281 |608.1711781253981  |2      |[0.34325268882569504,0.17038655116895285,0.4863607600053521]|
|2.1156839049820433|507.50235072377325 |2   